# Warehouse Analysis Notebook

This notebook loads the flattened Parquet tables from the Data Warehouse and displays them.

**Tables:** `genes`, `gene_seeds`, `powers`, `power_seeds`, `gene_regulation`, `gene_side_effects`.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
import pandas as pd

from super.core.runtime import bootstrap_spark_env
from super.core import utils
from super.core.display import display_scrollable_dataframe


In [ ]:

# Import shared warehouse logic
from super.core.warehouse import get_hero_genome_summary

In [ ]:
# Initialize Spark Session
bootstrap_spark_env()

spark = (SparkSession.builder
.appName("WarehouseViewer")
.config("spark.executor.memory", "4g")
.config("spark.driver.memory", "4g")
.getOrCreate())

In [ ]:
# Load Configuration
conf = utils.get_app_conf("generate_powers")
stage_root = conf.get_string("stage_root")
warehouse_root = os.path.join(stage_root, "warehouse")

print(f"Reading Warehouse from: {warehouse_root}")

## 1. Genes Table

In [ ]:
genes_df = spark.read.parquet(os.path.join(warehouse_root, "genes"))
display_scrollable_dataframe(genes_df.toPandas())

## 2. Hero Genomes (Unified View)
This cell reuses the logic from `analyze_heroes` (via `super.core.warehouse`) to show the full Master/Cluster view.

In [ ]:
full_report = get_hero_genome_summary(spark, warehouse_root)
display_scrollable_dataframe(full_report.orderBy("hero_name").toPandas())